<a href="https://colab.research.google.com/github/ShiX27/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
print("Total revenue:", df['revenue'].sum())
print("Total units:", df['qty'].sum())
df

Total revenue: 8520.0
Total units: 783


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5
...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0
396,V-01,Merch,2,24.0,48.0
397,V-10,Food,3,7.5,22.5
398,V-18,Merch,2,24.0,48.0


The total revenue is $8,520.00, and the total number of units is 783. The dataset contains 400 rows.

**Code Explanation:** First, calculate the revenue for each order by multiplying "qty" by "price". Then, use .sum() to calculate and print the total revenue and total number of units across all orders. Finally, display the whole table.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')['revenue'].sum().reset_index()
by_category['share_percent'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(2)
by_category = by_category.sort_values('revenue', ascending=False)
by_category

,category,revenue,share_percent
1,Food,4293.0,50.39
2,Merch,1771.5,20.79
0,Drink,1554.0,18.24
3,RainGear,901.5,10.58


Food has the the highest share of total revenue at 50.39%, while RainGear has the lowest share at 10.58%.

**Code Explanation:** First, group the "df" table by "category", calculate the total revenue for each category using .sum(), use .reset_index() to turn "category" from an index into a column, and store the result in "by_category". Second, create a new column called "share_percent" and calculate each category's share of total revenue by dividing each category's revenue by the total revenue across all categories, multiplying by 100, and rounding to two decimal places. Third, sort the table by revenue in desecending order, from highest to lowest. Finally, display the "by_category" table.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
by_vendor = df.groupby('vendor_id').agg(
    avg_revenue=('revenue', 'mean'),
    num_orders=('qty', 'count')
).round(2)
by_vendor = by_vendor.sort_values('avg_revenue', ascending=False)
by_vendor

,avg_revenue,num_orders
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


V-01 has the highest average order revenue at $22.60, with 94 orders.

**Code Explanation:** First, group the "df" table by "vendor_id" and use .agg() to calculate the average order revenue, stored as "avg_revenue" using 'mean', and the order count for each vendor, stored as "num_orders" using 'count'. Then, round "avg_revenue" to two decimal places and stored the grouped result in "by_vendor". Second, sort the table by "avg_revenue" in descending order, from highest to lowest. Finally, display the "by_vendor" table.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = by_category.loc[by_category['category'] == 'Merch', 'share_percent'].iloc[0]
print(f"{merch_share:.1f}%")

20.8%


The share of revenue comes from Merch is 20.8%.

**Code Explanation:** First, take the "by_category" table created in Q2 and use .loc[] to find the row where category is Merch. Then, select the "share_percent" value from the first row of the filtered result using .iloc[0] and store it as merch_share. Finally, print "merch_share" as a percentage rounded to one decimal place.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one', indicator=True)
print(f'Before: {len(df)} rows, ${df["revenue"].sum():.2f}')
print(f'After: {len(joined)} rows, ${joined['revenue'].sum():.2f}')
print(f'Unmatched vendor: {joined[joined["_merge"] == "left_only"]["vendor_id"].iloc[0]}')

# Check toal orders and revenue for the unmatched vendor
missing = joined[joined['_merge'] == 'left_only']
print(f'unmatched orders: {len(missing)} | '
      f'revenue at stake: ${missing["revenue"].sum():.2f}')

# Label vendor_name for V18 as 'Unknown vendor (V-18)'
joined = joined.drop(columns=['_merge'])
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor (V-18)')
joined

Before: 400 rows, $8520.00
After: 400 rows, $8520.00
Unmatched vendor: V-18
unmatched orders: 108 | revenue at stake: $2349.00


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown vendor (V-18)
2,V-18,Drink,3,4.5,13.5,Unknown vendor (V-18)
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown vendor (V-18)
...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unknown vendor (V-18)
396,V-01,Merch,2,24.0,48.0,Hoos Burgers
397,V-10,Food,3,7.5,22.5,Cav Merch North
398,V-18,Merch,2,24.0,48.0,Unknown vendor (V-18)


**The unmatched vendor, and what I did about it:** The unmatched vendor is V-18. It accounts for 108 orders and $2,349 in revenue, which is about 27.6% of total revenue. So, I kept its orders and labeled the vendor_name as "Unkown vendor (V-18)" rather than removing them.

I did that because I didn't want to drop the $2,349 in revenue that was actually earned, since removing it would understate the total revenue. Using a clear label like "Unknown vendor (V-18)" instead of keeping it as NaN makes the unmatched vendor easier to understand and helps identify what needs to be reviewed or corrected.

The row count and total revenue did not change after the merge. Both tables have 400 rows and $8,520 in total revenue.

**Code Explanation:** First, merge the "vendor_names" table with the "df" table using "vendor_id" and a left join. Use *validate='many_to_one'* to check that many rows in the "df" table can match one vendor in the "vendor_names" table. Use indicator=True to add a "_merge" column that shows where each row came from after the merge, and store the merged result in "joined". Second, print the number of rows and total revenue before and after the merge to prove that the row count and revenue total did not change. Third, print the unmatched vendor by finding the row where "the _merge" column is left_only and selecting its vendor_id. Fourth, check the total number of orders and revenue for the unmatched vendor. Fifth, drop the "_merge" column and label the "vendor_name" for V-18 as 'Unknown vendor (V-18)'. Finally, display the "joined" table.


### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot_table = joined.pivot_table(
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
)
pivot_table

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


**Code Explanation:** First, use the "joined" table created in Q5 to build a pivot table using .pivot_table(). Store 'revenue' in 'values' so it appears in the cells, store 'vendor_name' in 'index' so vendors appear as rows down the side, and store 'category' in 'columns' so categories appear across the top. Then, use aggfunc='sum' so each cell shows the sum of revenue. Second, use margins=True to add a total row and total column, and name them "Total". Finally, display the pivot table.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) For the next game, I would tell these vendors to focus more on Food because it made up the highest share of total revenue at 50.39%, which was much higher than other categories. RainGear made up only 10.58% of total revenue, so vendors could bring less RainGear unless the weather is expected to be bad. Merch made up 20.8% of total revenue, so vendors should still bring a good amount of merch, but Food should be the main focus.

b) Q5 is the least trustworthy answer because it includes an unmatched vendor, V-18, that could not be matched to a vendor name after the merge. V-18 had the highest number of orders and the highest total revenue among the four vendors, so having it as an unmatched vendor makes that part of the report less complete and less trustworthy.